<a href="https://colab.research.google.com/github/keivernunez/dataminingavanzado_austral/blob/main/Clase4/Note2_de_2_Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modelos Transformer para Traducción Automática

En este cuaderno se explora la implementación de un modelo Transformer para la traducción automática de portugués a inglés. Se utiliza un conjunto de datos paralelo y se implementan componentes clave como la codificación posicional, la atención multi-cabeza y las capas de encoder y decoder.

## Objetivos de Aprendizaje
- Comprender la arquitectura Transformer y sus ventajas sobre modelos recurrentes.
- Implementar codificación posicional y mecanismos de atención.
- Construir un encoder y decoder para tareas de traducción.
- Preprocesar datos de texto para modelado secuencia a secuencia.
- Entrenar y evaluar un modelo Transformer.
- Realizar inferencia para generar traducciones.

**Nota:** Este cuaderno está diseñado con fines didácticos. Cada sección incluye explicaciones detalladas, fragmentos de código comentados y visualizaciones para facilitar la comprensión. Basado en el trabajo seminal "Attention is All You Need" (Vaswani et al., 2017), se incorporan prácticas recomendadas para el procesamiento de lenguaje natural.

**Conceptos Clave a Recordar:**
- **Transformer:** Arquitectura basada en atención que procesa secuencias en paralelo, superando limitaciones de modelos RNN/LSTM.
- **Atención Multi-Cabeza:** Mecanismo que permite capturar múltiples relaciones semánticas simultáneamente.
- **Codificación Posicional:** Agrega información de posición a las incrustaciones para preservar el orden secuencial.

**Pregunta para Reflexión:** ¿Por qué la paralelización en Transformers mejora la eficiencia computacional en comparación con modelos recurrentes?

## 1. Importación de Bibliotecas

**Explicación:** Se importan bibliotecas esenciales para el manejo de datos, modelado y visualización. TensorFlow y Keras facilitan la construcción de capas personalizadas, mientras que Matplotlib y Seaborn permiten analizar resultados.

In [ ]:
# Importación de bibliotecas para manejo numérico y datos
import numpy as np  # Para operaciones con arreglos
import tensorflow as tf  # Framework principal de aprendizaje profundo
import tensorflow_datasets as tfds  # Para carga de conjuntos de datos
from tensorflow.keras.layers import Layer, Dense, Dropout, LayerNormalization  # Capas base
from tensorflow.keras.optimizers import Adam  # Optimizador
import matplotlib.pyplot as plt  # Para visualizaciones
import seaborn as sns  # Para gráficos estadísticos

## 2. Carga del Conjunto de Datos

**Explicación:** Se carga el conjunto TED Talks para traducción portugués-inglés. Este corpus paralelo es ideal para tareas seq2seq.

**Pregunta para Reflexión:** ¿Qué desafíos presenta un corpus multilingüe en términos de preprocesamiento?

In [ ]:
# Carga del conjunto de datos
ejemplos, metadatos = tfds.load('ted_hrlr_translate/pt_to_en', with_info=True, as_supervised=True)
ejemplos_entrenamiento, ejemplos_validación = ejemplos['train'], ejemplos['validation']

## 3. Tokenización

**Explicación:** La tokenización convierte texto en secuencias numéricas. Aquí se usan tokenizadores preentrenados para portugués e inglés.

In [ ]:
import types
# Tokenizadores para portugués e inglés
tokenizadores = types.SimpleNamespace()

tokenizadores.pt = tfds.deprecated.text.SubwordTextEncoder.build_from_corpus(
    (pt.numpy() for pt, en in ejemplos_entrenamiento), target_vocab_size=2**13)

tokenizadores.en = tfds.deprecated.text.SubwordTextEncoder.build_from_corpus(
    (en.numpy() for pt, en in ejemplos_entrenamiento), target_vocab_size=2**13)

## 4. Preparación de Lotes

**Explicación:** Se preparan lotes para entrenamiento eficiente, incluyendo relleno y desplazamiento para teacher forcing.

In [ ]:
import tensorflow as tf

# Hiperparámetros para lotes
TAMAÑO_LOTE = 64
TAMAÑO_BUFFER = 20000
LONGITUD_MÁX = 40

# Función para codificar ejemplos individuales (con truncado y tokens de inicio/fin)
def encode(pt, en):
    # Asumiendo que los tokenizadores no incluyen <start>/<end>; ajústalo si es necesario
    pt = [tokenizadores.pt.vocab_size] + tokenizadores.pt.encode(pt.numpy().decode('utf-8'))[:LONGITUD_MÁX] + [tokenizadores.pt.vocab_size + 1]
    en = [tokenizadores.en.vocab_size] + tokenizadores.en.encode(en.numpy().decode('utf-8'))[:LONGITUD_MÁX] + [tokenizadores.en.vocab_size + 1]
    return pt, en

# Envoltura para usar en tf.data (permite longitudes variables)
def tf_encode(pt, en):
    result_pt, result_en = tf.py_function(encode, [pt, en], [tf.int64, tf.int64])
    result_pt.set_shape([None])
    result_en.set_shape([None])
    return result_pt, result_en

# Función para preparar lotes después del padding (teacher forcing para decoder)
def preparar_lote(pt, en):
    # en incluye <start> y <end>; shift para inputs y labels
    entradas_en = en[:, :-1]
    etiquetas_en = en[:, 1:]
    return (pt, entradas_en), etiquetas_en

# Función para crear lotes (codificar primero, luego batch con padding)
def crear_lotes(ds):
    return (
        ds
        .map(tf_encode, num_parallel_calls=tf.data.AUTOTUNE)  # Codificar ejemplos individuales
        .filter(lambda pt, en: tf.size(pt) <= LONGITUD_MÁX + 2 and tf.size(en) <= LONGITUD_MÁX + 2)  # Filtrar por longitud (+2 por tokens start/end)
        .cache()  # Opcional: cache para speedup si el dataset cabe en memoria
        .shuffle(TAMAÑO_BUFFER)
        .padded_batch(TAMAÑO_LOTE, padded_shapes=([None], [None]), padding_values=(tf.constant(0, dtype=tf.int64), tf.constant(0, dtype=tf.int64)))  # Padding con 0, dtype int64
        .map(preparar_lote, num_parallel_calls=tf.data.AUTOTUNE)
        .prefetch(buffer_size=tf.data.AUTOTUNE))

# Creación de lotes de entrenamiento y validación
lotes_entrenamiento = crear_lotes(ejemplos_entrenamiento)
lotes_validacion = crear_lotes(ejemplos_validación)

## 5. Codificación Posicional

**Explicación:** La codificación posicional inyecta información de orden en las incrustaciones, muy importante, ya que Transformers no tienen recurrencia.

In [ ]:
# Función para codificación posicional
def codificación_posicional(longitud, profundidad):
    profundidad = profundidad / 2
    posiciones = np.arange(longitud)[:, np.newaxis]  # (sec, 1)
    profundidades = np.arange(profundidad)[np.newaxis, :] / profundidad  # (1, prof)
    tasas_ángulo = 1 / (10000 ** profundidades)  # (1, prof)
    rads_ángulo = posiciones * tasas_ángulo  # (pos, prof)

    cod_pos = np.concatenate([np.sin(rads_ángulo), np.cos(rads_ángulo)], axis=-1)

    return tf.cast(cod_pos, dtype=tf.float32)

# Capa de incrustación posicional
class IncrustacionPosicional(tf.keras.layers.Layer):
    def __init__(self, tamaño_vocab, d_modelo):
        super().__init__()
        self.d_modelo = d_modelo
        self.incrustacion = tf.keras.layers.Embedding(tamaño_vocab, d_modelo, mask_zero=True)
        self.cod_pos = codificación_posicional(longitud=2048, profundidad=d_modelo)

    def compute_mask(self, *args, **kwargs):
        return self.incrustacion.compute_mask(*args, **kwargs)  # Máscara de relleno

    def call(self, x):
        longitud = tf.shape(x)[1]
        x = self.incrustacion(x)
        x *= tf.math.sqrt(tf.cast(self.d_modelo, tf.float32))  # Escalado
        x = x + self.cod_pos[tf.newaxis, :longitud, :]  # Adición de codificación
        return x

## 6. Capa de Atención Base

**Explicación:** Esta capa implementa la atención escalada por punto, base para multi-cabeza.

In [ ]:
# Capa de atención base
class AtenciónBase(Layer):
    def __init__(self, **kwargs):
        super().__init__()
        self.atención_mha = tf.keras.layers.MultiHeadAttention(**kwargs)  # Atención multi-cabeza
        self.norm_capa = LayerNormalization()  # Normalización
        self.suma = tf.keras.layers.Add()  # Suma residual

    def call(self, x, contexto):
        attn = self.atención_mha(query=x, value=contexto)  # Cálculo de atención
        x = self.suma([x, attn])  # Conexión residual
        x = self.norm_capa(x)  # Normalización
        return x

## 7. Capa de Atención Cruzada

**Explicación:** Permite que el decoder atienda al output del encoder.

In [ ]:
# Capa de atención cruzada
class AtenciónCruzada(AtenciónBase):
    def call(self, x, contexto):
        salida_atn, puntuaciones_atn = self.atención_mha(
            query=x, key=contexto, value=contexto, return_attention_scores=True)  # Atención con puntuaciones
        self.últimas_puntuaciones_atn = puntuaciones_atn  # Almacenamiento para visualización
        x = self.suma([x, salida_atn])
        x = self.norm_capa(x)
        return x

## 8. Capa de Atención Global

**Explicación:** Atención auto-atención en el encoder.

In [ ]:
# Capa de atención global (auto-atención)
class AtenciónGlobal(AtenciónBase):
    def call(self, x):
        salida_atn = self.atención_mha(query=x, value=x, key=x)  # Auto-atención
        x = self.suma([x, salida_atn])
        x = self.norm_capa(x)
        return x

## 9. Capa de Atención Causal

**Explicación:** Asegura que el decoder solo atienda a tokens previos durante la generación.

In [ ]:
# Capa de atención causal
class AtenciónCausal(AtenciónBase):
    def call(self, x):
        salida_atn = self.atención_mha(query=x, value=x, key=x, use_causal_mask=True)  # Máscara causal
        x = self.suma([x, salida_atn])
        x = self.norm_capa(x)
        return x

## 10. Capa de Red de Alimentación Adelante

**Explicación:** Capa fully connected para procesar características locales.

In [ ]:
# Capa de red de alimentación adelante
class RedAlimentaciónAdelante(Layer):
    def __init__(self, d_modelo, dff, tasa_dropout=0.1):
        super().__init__()
        self.sec_densa = tf.keras.Sequential([
            tf.keras.layers.Dense(dff, activation='relu'),  # Capa oculta
            tf.keras.layers.Dense(d_modelo)  # Proyección de salida
        ])
        self.suma = tf.keras.layers.Add()  # Conexión residual
        self.norm_capa = LayerNormalization()  # Normalización
        self.dropout = Dropout(tasa_dropout)  # Regularización

    def call(self, x, entrenamiento=False):
        x = self.suma([x, self.sec_densa(x)])  # Residual
        x = self.norm_capa(x)
        x = self.dropout(x, training=entrenamiento)
        return x

## 11. Capa de Encoder

**Explicación:** Compuesta por múltiples subcapas de atención y feed-forward.

In [ ]:
# Capa de encoder
class CapaEncoder(tf.keras.layers.Layer):
    def __init__(self, d_modelo, num_cabezas, dff, tasa_dropout=0.1):
        super().__init__()
        self.atencion_multi = tf.keras.layers.MultiHeadAttention(num_heads=num_cabezas, key_dim=d_modelo)
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(dff, activation='relu'),
            tf.keras.layers.Dense(d_modelo)
        ])
        self.norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(tasa_dropout)
        self.dropout2 = tf.keras.layers.Dropout(tasa_dropout)

    def call(self, x, entrenamiento=False):
        attn_output = self.atencion_multi(x, x, x)
        attn_output = self.dropout1(attn_output, training=entrenamiento)
        out1 = self.norm1(x + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=entrenamiento)
        out2 = self.norm2(out1 + ffn_output)
        return out2

## 12. Capa de Decoder

**Explicación:** Incluye atención causal, cruzada y feed-forward.

In [ ]:
# Capa de decoder
class CapaDecoder(tf.keras.layers.Layer):
    def __init__(self, d_modelo, num_cabezas, dff, tasa_dropout=0.1):
        super().__init__()
        self.atencion1 = tf.keras.layers.MultiHeadAttention(num_heads=num_cabezas, key_dim=d_modelo)
        self.atencion2 = tf.keras.layers.MultiHeadAttention(num_heads=num_cabezas, key_dim=d_modelo)
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(dff, activation='relu'),
            tf.keras.layers.Dense(d_modelo)
        ])
        self.norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.norm3 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(tasa_dropout)
        self.dropout2 = tf.keras.layers.Dropout(tasa_dropout)
        self.dropout3 = tf.keras.layers.Dropout(tasa_dropout)

    def call(self, x, contexto, entrenamiento=False):
        attn1 = self.atencion1(x, x, x)
        attn1 = self.dropout1(attn1, training=entrenamiento)
        out1 = self.norm1(attn1 + x)

        attn2 = self.atencion2(out1, contexto, contexto)
        attn2 = self.dropout2(attn2, training=entrenamiento)
        out2 = self.norm2(attn2 + out1)

        ffn_output = self.ffn(out2)
        ffn_output = self.dropout3(ffn_output, training=entrenamiento)
        out3 = self.norm3(ffn_output + out2)
        return out3

## 13. Modelo Encoder

**Explicación:** Apila múltiples capas de encoder.

In [ ]:
# Modelo encoder completo
class Encoder(tf.keras.layers.Layer):
    def __init__(self, num_capas, d_modelo, num_cabezas, dff, tamaño_vocab, tasa_dropout=0.1):
        super().__init__()
        self.num_capas = num_capas
        self.d_modelo = d_modelo
        self.embedding = tf.keras.layers.Embedding(tamaño_vocab, d_modelo)
        self.dropout = tf.keras.layers.Dropout(tasa_dropout)
        self.capas_enc = [CapaEncoder(d_modelo, num_cabezas, dff, tasa_dropout)
                          for _ in range(num_capas)]

    def call(self, x, entrenamiento=False):
        x = self.embedding(x)
        x *= tf.math.sqrt(tf.cast(self.d_modelo, tf.float32))
        x = self.dropout(x, training=entrenamiento)
        for capa in self.capas_enc:
            x = capa(x, entrenamiento=entrenamiento)
        return x

## 14. Modelo Decoder

**Explicación:** Apila múltiples capas de decoder.

In [ ]:
# Modelo decoder completo
class Decoder(tf.keras.layers.Layer):
    def __init__(self, num_capas, d_modelo, num_cabezas, dff, tamaño_vocab, tasa_dropout=0.1):
        super().__init__()
        self.num_capas = num_capas
        self.d_modelo = d_modelo
        self.embedding = tf.keras.layers.Embedding(tamaño_vocab, d_modelo)
        self.dropout = tf.keras.layers.Dropout(tasa_dropout)
        self.capas_dec = [CapaDecoder(d_modelo, num_cabezas, dff, tasa_dropout)
                          for _ in range(num_capas)]

    def call(self, x, contexto, entrenamiento=False):
        x = self.embedding(x)
        x *= tf.math.sqrt(tf.cast(self.d_modelo, tf.float32))
        x = self.dropout(x, training=entrenamiento)
        for capa in self.capas_dec:
            x = capa(x, contexto, entrenamiento=entrenamiento)
        return x

## 15. Modelo Transformer Completo

**Explicación:** Integra encoder y decoder con una capa final de proyección.

In [ ]:
# Clase Transformer
class Transformer(tf.keras.Model):
    def __init__(self, *, num_capas, d_modelo, num_cabezas, dff, tamaño_vocab_entrada, tamaño_vocab_objetivo, tasa_dropout=0.1):
        super().__init__()
        self.encoder = Encoder(num_capas=num_capas, d_modelo=d_modelo, num_cabezas=num_cabezas, dff=dff, tamaño_vocab=tamaño_vocab_entrada, tasa_dropout=tasa_dropout)  # Encoder
        self.decoder = Decoder(num_capas=num_capas, d_modelo=d_modelo, num_cabezas=num_cabezas, dff=dff, tamaño_vocab=tamaño_vocab_objetivo, tasa_dropout=tasa_dropout)  # Decoder
        self.capa_final = Dense(tamaño_vocab_objetivo)  # Proyección a vocabulario

    def call(self, entradas, entrenamiento=False):
        contexto, x = entradas
        contexto = self.encoder(contexto, entrenamiento)  # Paso por encoder
        x = self.decoder(x, contexto, entrenamiento)  # Paso por decoder
        logits = self.capa_final(x)  # Logits de salida
        return logits

## 16. Optimizador Personalizado

**Explicación:** Optimizador con tasa de aprendizaje variable como en el paper original.

In [ ]:
# Optimizador personalizado
class OptimizadorPersonalizado(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, d_modelo, pasos_calentamiento=4000):
        super().__init__()
        self.d_modelo = d_modelo
        self.pasos_calentamiento = pasos_calentamiento

    def __call__(self, paso):
        paso = tf.cast(paso, dtype=tf.float32)
        arg1 = tf.math.rsqrt(paso)
        arg2 = paso * (self.pasos_calentamiento ** -1.5)
        return tf.math.rsqrt(tf.cast(self.d_modelo, tf.float32)) * tf.math.minimum(arg1, arg2)

## 17. Funciones de Pérdida y Precisión con Máscara

**Explicación:** Ignoran el relleno en cálculos para enfocarse en tokens válidos.

In [ ]:
# Pérdida con máscara
def perdida_enmascarada(etiquetas, predicciones):
    perdida_entropía_cruzada = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')
    perdida = perdida_entropía_cruzada(etiquetas, predicciones)
    mascara = tf.cast(etiquetas != 0, dtype=tf.float32)  # Máscara para relleno
    perdida *= mascara
    return tf.reduce_sum(perdida) / tf.reduce_sum(mascara)  # Pérdida promedio

# Precisión con máscara
def precisión_enmascarada(etiquetas, predicciones):
    predicciones = tf.argmax(predicciones, axis=-1)
    etiquetas = tf.cast(etiquetas, dtype=tf.int64)
    coincidencia = etiquetas == predicciones
    mascara = etiquetas != 0
    coincidencia = coincidencia & mascara
    coincidencia = tf.cast(coincidencia, dtype=tf.float32)
    mascara = tf.cast(mascara, dtype=tf.float32)
    return tf.reduce_sum(coincidencia) / tf.reduce_sum(mascara)  # Precisión promedio

## 18. Instanciación y Compilación del Modelo

**Explicación:** Se define el Transformer con hiperparámetros específicos.

In [ ]:
# Hiperparámetros del modelo
num_capas = 4
d_modelo = 128
dff = 512
num_cabezas = 8
tasa_dropout = 0.1

# Instanciación
transformer = Transformer(
    num_capas=num_capas,
    d_modelo=d_modelo,
    num_cabezas=num_cabezas,
    dff=dff,
    tamaño_vocab_entrada=tokenizadores.pt.vocab_size,
    tamaño_vocab_objetivo=tokenizadores.en.vocab_size,
    tasa_dropout=tasa_dropout)

## 19. Entrenamiento del Modelo

**Explicación:** Se entrena con pérdida enmascarada y optimizador personalizado.

# Utilizando GPU para mejorar los tiempos de procesamiento

In [ ]:
import tensorflow as tf

# ======================================================
# ⚙️ CONFIGURACIÓN GPU
# ======================================================

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU detectada y configurada: {gpus[0].name}")
    except RuntimeError as e:
        print(f"Error al inicializar GPU: {e}")
else:
    print("No se detectó GPU. TensorFlow usará CPU.")

# (Opcional pero recomendado para GPUs modernas como RTX 30xx o 40xx)
# Esto acelera el entrenamiento usando menor precisión
try:
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy('mixed_float16')
    print("✅ Política de precisión mixta activada (float16).")
except Exception as e:
    print(f"No se activó mixed precision: {e}")

# ======================================================
# 🔧 CLASES DEL MODELO
# ======================================================

class CapaEncoder(tf.keras.layers.Layer):
    def __init__(self, d_modelo, num_cabezas, dff, tasa_dropout=0.1):
        super().__init__()
        self.atencion_multi = tf.keras.layers.MultiHeadAttention(num_heads=num_cabezas, key_dim=d_modelo)
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(dff, activation='relu'),
            tf.keras.layers.Dense(d_modelo)
        ])
        self.norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(tasa_dropout)
        self.dropout2 = tf.keras.layers.Dropout(tasa_dropout)

    def call(self, x, entrenamiento=False):
        attn_output = self.atencion_multi(x, x, x)
        attn_output = self.dropout1(attn_output, training=entrenamiento)
        out1 = self.norm1(x + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=entrenamiento)
        return self.norm2(out1 + ffn_output)


class Encoder(tf.keras.layers.Layer):
    def __init__(self, num_capas, d_modelo, num_cabezas, dff, tamaño_vocab, tasa_dropout=0.1):
        super().__init__()
        self.num_capas = num_capas
        self.d_modelo = d_modelo
        self.embedding = tf.keras.layers.Embedding(tamaño_vocab, d_modelo)
        self.dropout = tf.keras.layers.Dropout(tasa_dropout)
        self.capas_enc = [CapaEncoder(d_modelo, num_cabezas, dff, tasa_dropout) for _ in range(num_capas)]

    def call(self, x, entrenamiento=False):
        x = self.embedding(x)
        x *= tf.math.sqrt(tf.cast(self.d_modelo, tf.float32))
        x = self.dropout(x, training=entrenamiento)
        for capa in self.capas_enc:
            x = capa(x, entrenamiento=entrenamiento)
        return x


class CapaDecoder(tf.keras.layers.Layer):
    def __init__(self, d_modelo, num_cabezas, dff, tasa_dropout=0.1):
        super().__init__()
        self.atencion1 = tf.keras.layers.MultiHeadAttention(num_heads=num_cabezas, key_dim=d_modelo)
        self.atencion2 = tf.keras.layers.MultiHeadAttention(num_heads=num_cabezas, key_dim=d_modelo)
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(dff, activation='relu'),
            tf.keras.layers.Dense(d_modelo)
        ])
        self.norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.norm3 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(tasa_dropout)
        self.dropout2 = tf.keras.layers.Dropout(tasa_dropout)
        self.dropout3 = tf.keras.layers.Dropout(tasa_dropout)

    def call(self, x, contexto, entrenamiento=False):
        attn1 = self.atencion1(x, x, x)
        attn1 = self.dropout1(attn1, training=entrenamiento)
        out1 = self.norm1(attn1 + x)

        attn2 = self.atencion2(out1, contexto, contexto)
        attn2 = self.dropout2(attn2, training=entrenamiento)
        out2 = self.norm2(attn2 + out1)

        ffn_output = self.ffn(out2)
        ffn_output = self.dropout3(ffn_output, training=entrenamiento)
        return self.norm3(ffn_output + out2)


class Decoder(tf.keras.layers.Layer):
    def __init__(self, num_capas, d_modelo, num_cabezas, dff, tamaño_vocab, tasa_dropout=0.1):
        super().__init__()
        self.num_capas = num_capas
        self.d_modelo = d_modelo
        self.embedding = tf.keras.layers.Embedding(tamaño_vocab, d_modelo)
        self.dropout = tf.keras.layers.Dropout(tasa_dropout)
        self.capas_dec = [CapaDecoder(d_modelo, num_cabezas, dff, tasa_dropout) for _ in range(num_capas)]

    def call(self, x, contexto, entrenamiento=False):
        x = self.embedding(x)
        x *= tf.math.sqrt(tf.cast(self.d_modelo, tf.float32))
        x = self.dropout(x, training=entrenamiento)
        for capa in self.capas_dec:
            x = capa(x, contexto, entrenamiento=entrenamiento)
        return x


class Transformer(tf.keras.Model):
    def __init__(self, *, num_capas, d_modelo, num_cabezas, dff, vocab_entrada, vocab_salida, tasa_dropout=0.1):
        super().__init__()
        self.encoder = Encoder(num_capas, d_modelo, num_cabezas, dff, vocab_entrada, tasa_dropout)
        self.decoder = Decoder(num_capas, d_modelo, num_cabezas, dff, vocab_salida, tasa_dropout)
        self.capa_final = tf.keras.layers.Dense(vocab_salida)

    def call(self, entradas, training=False):
        contexto, x = entradas
        contexto = self.encoder(contexto, entrenamiento=training)
        x = self.decoder(x, contexto, entrenamiento=training)
        return self.capa_final(x)


# ======================================================
# ⚡ OPTIMIZADOR GPU-FRIENDLY
# ======================================================

class OptimizadorPersonalizado(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, d_modelo, pasos_calentamiento=4000):
        super().__init__()
        self.d_modelo = tf.cast(d_modelo, tf.float32)
        self.pasos_calentamiento = pasos_calentamiento

    def __call__(self, paso):
        paso = tf.cast(paso, tf.float32)
        arg1 = tf.math.rsqrt(paso)
        arg2 = paso * (self.pasos_calentamiento ** -1.5)
        return tf.math.rsqrt(self.d_modelo) * tf.math.minimum(arg1, arg2)


# ======================================================
# 🚀 INICIALIZACIÓN Y ENTRENAMIENTO
# ======================================================

# Suponiendo que ya tienes definidos:
# TAMAÑO_LOTE, LONGITUD_MÁX, tokenizadores, perdida_enmascarada, precisión_enmascarada, lotes_entrenamiento, lotes_validación

transformer = Transformer(
    num_capas=4,
    d_modelo=128,
    num_cabezas=8,
    dff=512,
    vocab_entrada=tokenizadores.pt.vocab_size + 2,
    vocab_salida=tokenizadores.en.vocab_size + 2
)

# Ejemplo de construcción explícita
entrada_ejemplo_pt = tf.zeros((TAMAÑO_LOTE, LONGITUD_MÁX + 2), dtype=tf.int64)
entrada_ejemplo_en = tf.zeros((TAMAÑO_LOTE, LONGITUD_MÁX + 1), dtype=tf.int64)
_ = transformer((entrada_ejemplo_pt, entrada_ejemplo_en), training=False)

# Compilación con optimizador en GPU
optimizador = tf.keras.optimizers.Adam(
    OptimizadorPersonalizado(d_modelo=128),
    beta_1=0.9, beta_2=0.98, epsilon=1e-9
)

transformer.compile(
    loss=perdida_enmascarada,
    optimizer=optimizador,
    metrics=[precisión_enmascarada]
)

# Entrenamiento (usa GPU automáticamente si está disponible)
transformer.fit(lotes_entrenamiento, epochs=1, validation_data=lotes_validación)


# Sin utilizar GPU

In [ ]:
# =============================
# TRANSFORMER
# =============================

class Transformer(tf.keras.Model):
    def __init__(self, *, num_capas, d_modelo, num_cabezas, dff,
                 vocab_entrada, vocab_salida, tasa_dropout=0.1):
        super().__init__()
        self.encoder = Encoder(num_capas=num_capas, d_modelo=d_modelo,
                               num_cabezas=num_cabezas, dff=dff,
                               tamaño_vocab=vocab_entrada, tasa_dropout=tasa_dropout)
        self.decoder = Decoder(num_capas=num_capas, d_modelo=d_modelo,
                               num_cabezas=num_cabezas, dff=dff,
                               tamaño_vocab=vocab_salida, tasa_dropout=tasa_dropout)
        self.capa_final = tf.keras.layers.Dense(vocab_salida)

    def call(self, entradas, training=False):
        contexto, x = entradas
        contexto = self.encoder(contexto, entrenamiento=training)
        x = self.decoder(x, contexto, entrenamiento=training)
        logits = self.capa_final(x)
        return logits


# =============================
# OPTIMIZADOR PERSONALIZADO
# =============================

class OptimizadorPersonalizado(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, d_modelo, pasos_calentamiento=4000):
        super().__init__()
        self.d_modelo = tf.cast(d_modelo, tf.float32)
        self.pasos_calentamiento = pasos_calentamiento

    def __call__(self, paso):
        paso = tf.cast(paso, tf.float32)
        arg1 = tf.math.rsqrt(paso)
        arg2 = paso * (self.pasos_calentamiento ** -1.5)
        return tf.math.rsqrt(self.d_modelo) * tf.math.minimum(arg1, arg2)


# =============================
# INICIALIZACIÓN DEL MODELO
# =============================

# Suponiendo que tienes definidas:
# TAMAÑO_LOTE, LONGITUD_MÁX, tokenizadores, perdida_enmascarada, precisión_enmascarada, lotes_entrenamiento, lotes_validación

transformer = Transformer(
    num_capas=4,
    d_modelo=128,
    num_cabezas=8,
    dff=512,
    vocab_entrada=tokenizadores.pt.vocab_size + 2,
    vocab_salida=tokenizadores.en.vocab_size + 2
)

# Construcción explícita
entrada_ejemplo_pt = tf.zeros((TAMAÑO_LOTE, LONGITUD_MÁX + 2), dtype=tf.int64)
entrada_ejemplo_en = tf.zeros((TAMAÑO_LOTE, LONGITUD_MÁX + 1), dtype=tf.int64)
transformer((entrada_ejemplo_pt, entrada_ejemplo_en), training=False)
transformer.build([(None, LONGITUD_MÁX + 2), (None, LONGITUD_MÁX + 1)])

# Compilación
optimizador = tf.keras.optimizers.Adam(
    OptimizadorPersonalizado(d_modelo=128),
    beta_1=0.9, beta_2=0.98, epsilon=1e-9
)

transformer.compile(
    loss=perdida_enmascarada,
    optimizer=optimizador,
    metrics=[precisión_enmascarada]
)

# Entrenamiento
transformer.fit(lotes_entrenamiento, epochs=1, validation_data=lotes_validación)


## 20. Inferencia con el Traductor

**Explicación:** Generación autoregresiva de traducciones token por token.

In [ ]:
# Clase para traducción
class Traductor(tf.Module):
    def __init__(self, tokenizadores, transformer):
        self.tokenizadores = tokenizadores
        self.transformer = transformer

    def __call__(self, oración, longitud_máx=128):
        # Tokenización de entrada
        entrada_encoder = self.tokenizadores.pt.tokenize(oración).to_tensor()

        # Tokens de inicio y fin
        inicio_fin = self.tokenizadores.en.tokenize([''])[0]
        inicio = inicio_fin[0][tf.newaxis]
        fin = inicio_fin[1][tf.newaxis]

        arreglo_salida = tf.TensorArray(dtype=tf.int64, size=0, dynamic_size=True)
        arreglo_salida = arreglo_salida.write(0, inicio)

        for i in tf.range(longitud_máx):
            salida = tf.transpose(arreglo_salida.stack())
            predicciones = self.transformer([entrada_encoder, salida], training=False)
            predicciones = predicciones[:, -1:, :]  # Último token
            id_predicho = tf.argmax(predicciones, axis=-1)
            arreglo_salida = arreglo_salida.write(i+1, id_predicho[0])
            if id_predicho == fin:
                break

        salida = tf.transpose(arreglo_salida.stack())
        texto = tokenizadores.en.detokenize(salida)[0]  # Detokenización
        tokens = tokenizadores.en.lookup(salida)[0]  # Tokens

        return texto, tokens

# Instanciación del traductor
traductor = Traductor(tokenizadores, transformer)

# Ejemplo de uso
oración = tf.constant('este é um problema que temos que resolver.')
texto_traducido, tokens_traducidos = traductor(oración)

print(f'Oración: {oración.numpy().decode("utf-8")}')
print(f'Traducción: {texto_traducido.numpy().decode("utf-8")}')

## Cierre del Notebook

Se presentó una implementación completa de un Transformer para traducción. Para mejoras, considere agregar más capas o usar pre-entrenamiento.

**Actividad Sugerida:** Modificar el número de cabezas de atención y observe el impacto en la precisión.